# `01_data_prep.ipynb` — Data Preparation
**Project:** Cross-lingual RAG for Indonesian–English QA  
**Phase:** Pre-experiment  
**Kaggle accelerator:** CPU only (no GPU needed)  
**Est. runtime:** ~30–60 min (dominated by Wikipedia ID streaming)

---

### What this notebook does
| Step | Task | Output |
|------|------|--------|
| 1 | Install dependencies | — |
| 2 | Load HotpotQA (train + eval) | `hotpotqa_train.jsonl`, `hotpotqa_eval.jsonl`, `hotpotqa_source_docs.jsonl` |
| 3 | Load TyDiQA-ID, stratified 80/20 split | `tydiqa_id_train.jsonl`, `tydiqa_id_eval.jsonl` |
| 4 | Load XQuAD-ID (1,190 full), preserve parallel EN/ID structure | `xquad_id_parallel.jsonl` |
| 5 | Stream Wikipedia ID, sample 20K articles | `wikipedia_id_corpus.jsonl` |
| 6 | Validate all outputs | shape checks, field checks |

> ⚠️ **Persist outputs:** After this notebook finishes, upload everything in `/kaggle/working/data/` to the Kaggle Dataset `crosslingual-rag-data`. All downstream notebooks (NB02–NB08) read from that dataset at `/kaggle/input/crosslingual-rag-data/`.


## 1. Install Dependencies

In [ ]:
# Only packages not pre-installed on Kaggle
!pip install -q datasets huggingface_hub

import importlib, sys
for pkg in ["datasets", "huggingface_hub"]:
    assert importlib.import_module(pkg), f"Import failed: {pkg}"
print("✓ All packages ready")


## 2. Imports & Output Paths

In [ ]:
import os
import re
import json
import random
import collections
from pathlib import Path

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)

# Hugging Face auth from Kaggle Secrets.
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
HF_TOKEN = secret_value_0

# ── Output directory ─────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset sizes (targets) ───────────────────────────────────────────────
HOTPOTQA_TRAIN_TARGET = 10_000   # sample from full distractor train split
HOTPOTQA_EVAL_TARGET  = 7_405    # full validation split
WIKI_ID_TARGET        = 20_000   # streamed from Wikipedia dump
TYDIQA_TRAIN_RATIO    = 0.80     # stratified split for TyDiQA-ID

print(f"Output dir: {DATA_DIR}")
print(f"Seed: {SEED}")
print(f"Hugging Face token: {'configured' if HF_TOKEN else 'not configured'}")


## 3. Utility Functions

In [ ]:
def write_jsonl(records, path):
    """Write list of dicts to .jsonl, return count written."""
    path = Path(path)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    count = len(records)
    print(f"  → Wrote {count:,} records to {path.name}")
    return count


def read_jsonl(path):
    """Read .jsonl back into a list of dicts (for validation)."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def stratified_split(records, key, ratio, seed=SEED):
    """
    Split `records` into (train, eval) with `ratio` going to train.
    Stratified by `key` field so each stratum is split proportionally.
    Returns (train_list, eval_list).
    """
    rng = random.Random(seed)
    buckets = collections.defaultdict(list)
    for rec in records:
        buckets[rec[key]].append(rec)

    train, eval_ = [], []
    for label, items in buckets.items():
        rng.shuffle(items)
        split_idx = max(1, int(len(items) * ratio))
        train.extend(items[:split_idx])
        eval_.extend(items[split_idx:])

    rng.shuffle(train)
    rng.shuffle(eval_)
    return train, eval_


print("✓ Utilities defined")


## 4. HotpotQA

Three outputs:
- `hotpotqa_train.jsonl` — ~10K instances (sampled from distractor train split) for classifier training in NB02. All get label **C**.
- `hotpotqa_eval.jsonl` — full validation split (7,405 instances) for ablation in NB06.
- `hotpotqa_source_docs.jsonl` — supporting document passages used as the retrieval corpus in NB03/NB06.

**Unified record format:** `{id, question, answer, lang, dataset, split}`  
**Source doc format:** `{doc_id, title, text, dataset}`


In [ ]:
from datasets import load_dataset

print("Loading HotpotQA (distractor config)…")
hotpotqa_raw = load_dataset("hotpot_qa", "distractor", trust_remote_code=True)
print(f"  Train: {len(hotpotqa_raw['train']):,} | Validation: {len(hotpotqa_raw['validation']):,}")


In [ ]:
# ── hotpotqa_eval (full validation split) ────────────────────────────────
hotpotqa_eval_records = []
for ex in hotpotqa_raw["validation"]:
    hotpotqa_eval_records.append({
        "id":      ex["id"],
        "question": ex["question"],
        "answer":  ex["answer"],
        "lang":    "en",
        "dataset": "hotpotqa",
        "split":   "eval",
    })

write_jsonl(hotpotqa_eval_records, DATA_DIR / "hotpotqa_eval.jsonl")
assert len(hotpotqa_eval_records) == HOTPOTQA_EVAL_TARGET, (
    f"Expected {HOTPOTQA_EVAL_TARGET} eval records, got {len(hotpotqa_eval_records)}"
)
print(f"  ✓ HotpotQA eval: {len(hotpotqa_eval_records):,} records")


In [ ]:
# ── hotpotqa_train (sample 10K from train split for NB02 classifier) ─────
rng = random.Random(SEED)
train_indices = list(range(len(hotpotqa_raw["train"])))
rng.shuffle(train_indices)
sampled_indices = train_indices[:HOTPOTQA_TRAIN_TARGET]

hotpotqa_train_records = []
for idx in sampled_indices:
    ex = hotpotqa_raw["train"][idx]
    hotpotqa_train_records.append({
        "id":      ex["id"],
        "question": ex["question"],
        "answer":  ex["answer"],
        "lang":    "en",
        "dataset": "hotpotqa",
        "split":   "train",
        # Label C hardcoded — multi-hop by design, no heuristic needed
        "complexity_label": "C",
    })

write_jsonl(hotpotqa_train_records, DATA_DIR / "hotpotqa_train.jsonl")
print(f"  ✓ HotpotQA train: {len(hotpotqa_train_records):,} records (all label C)")


In [ ]:
# ── hotpotqa_source_docs (retrieval corpus for NB03/NB06) ─────────────────
# Each HotpotQA example has `context`: list of [title, sentences_list] pairs.
# Deduplicate by title across the FULL dataset (train + validation) so the
# retrieval corpus is complete and not eval-split-specific.

seen_titles = set()
source_docs = []

for split_name in ["train", "validation"]:
    for ex in hotpotqa_raw[split_name]:
        for title, sentences in zip(
            ex["context"]["title"],
            ex["context"]["sentences"]
        ):
            if title not in seen_titles:
                seen_titles.add(title)
                source_docs.append({
                    "doc_id":  f"hotpotqa__{title.replace(' ', '_')}",
                    "title":   title,
                    "text":    " ".join(sentences),
                    "dataset": "hotpotqa",
                })

write_jsonl(source_docs, DATA_DIR / "hotpotqa_source_docs.jsonl")
print(f"  ✓ HotpotQA source docs: {len(source_docs):,} unique documents")

# Free RAM — HotpotQA is large
del hotpotqa_raw
import gc; gc.collect()
print("  ✓ Released HotpotQA from memory")


## 5. TyDiQA-ID

Source: Indonesian JSONL file from `khalidalt/tydiqa-goldp`, loaded with the generic `json` loader to avoid the deprecated dataset script.

**Labeling heuristic (for NB02 classifier training):**
- **Label A** — answer word count ≤ 3 AND answer matches named-entity proxy pattern (number, date, or capitalized token)
- **Label B** — answer word count > 3 (single-span phrase/clause)
- **Discarded** — answers that are empty or don't fit A/B clearly (logged)

TyDiQA-ID produces only Class A and B. Class C comes exclusively from HotpotQA.

**Split:** 80/20 stratified by complexity label.
- `tydiqa_id_train.jsonl` — 80% for classifier training (NB02)
- `tydiqa_id_eval.jsonl` — 20% reserved for downstream evaluation (NB07/NB08)


In [ ]:
print("Loading TyDiQA-GoldP Indonesian JSONL...")

# Dataset scripts are no longer supported by datasets>=4.0, so load the
# Indonesian JSONL file directly with the generic JSON loader.
TYDIQA_ID_TRAIN_URL = "https://huggingface.co/datasets/khalidalt/tydiqa-goldp/resolve/main/train/indonesian-train.jsonl"
tydiqa_raw = load_dataset("json", data_files={"train": TYDIQA_ID_TRAIN_URL}, token=HF_TOKEN)

tydiqa_id_all = list(tydiqa_raw["train"])

if len(tydiqa_id_all) == 0:
    raise RuntimeError("Loaded Indonesian TyDiQA JSONL but found 0 samples")

print(f"  Found {len(tydiqa_id_all):,} Indonesian instances (TyDiQA-ID)")


In [ ]:
# ── Labeling heuristic ────────────────────────────────────────────────────

# Named-entity proxy: number (incl. year), date pattern, or capitalized word
NE_PATTERN = re.compile(
    r"^(?:"
    r"\d[\d,\.]*"               # number / decimal / comma-separated
    r"|\d{1,2}[\-/]\d{1,2}[\-/]\d{2,4}"  # date dd-mm-yyyy
    r"|[A-Z][a-zA-Z]+"             # capitalized token (proper noun proxy)
    r"|[A-Z]{2,}"                  # acronym
    r")$"
)

def classify_tydiqa(answer_text: str):
    """
    Returns 'A', 'B', or None (discard).
    None is logged but not written to any output file.
    """
    if not answer_text or not answer_text.strip():
        return None
    words = answer_text.strip().split()
    if len(words) <= 3:
        # Check each word against NE proxy pattern
        if any(NE_PATTERN.match(w) for w in words):
            return "A"
        # Short answer but no NE signal → borderline; treat as B to reduce discard rate
        # Paper note: borderline short answers assigned B, not discarded
        return "B"
    return "B"


label_counts = {"A": 0, "B": 0, "discarded": 0}
labeled_records = []

for ex in tydiqa_id_all:
    # TyDiQA secondary task stores the answer in 'answers' dict
    answer_text = ""
    if "answers" in ex and ex["answers"]:
        # Can be dict with 'text' list or just a list
        ans = ex["answers"]
        if isinstance(ans, dict) and "text" in ans:
            texts = ans["text"]
            answer_text = texts[0] if texts else ""
        elif isinstance(ans, list) and len(ans) > 0:
            first = ans[0]
            answer_text = first.get("text", first) if isinstance(first, dict) else str(first)

    label = classify_tydiqa(answer_text)

    if label is None:
        label_counts["discarded"] += 1
        continue

    label_counts[label] += 1
    labeled_records.append({
        "id":               ex.get("id", f"tydiqa_{len(labeled_records)}"),
        "question":         ex.get("question", ex.get("question_text", "")),
        "answer":           answer_text,
        "lang":             "id",
        "dataset":          "tydiqa_id",
        "split":            "train",     # will be overwritten after split
        "complexity_label": label,
    })

print(f"  Label distribution:")
total_seen = sum(label_counts.values())
for k, v in label_counts.items():
    print(f"    {k}: {v:,}  ({v/total_seen*100:.1f}%)")
print(f"  Total usable records: {len(labeled_records):,}")

# Log discard count for paper
discard_log = {
    "total_tydiqa_id": len(tydiqa_id_all),
    "label_A": label_counts["A"],
    "label_B": label_counts["B"],
    "discarded": label_counts["discarded"],
    "discard_rate_pct": round(label_counts["discarded"] / len(tydiqa_id_all) * 100, 2),
    "seed": SEED,
}
with open(DATA_DIR / "tydiqa_labeling_log.json", "w") as f:
    json.dump(discard_log, f, indent=2)
print(f"  ✓ Discard log saved → tydiqa_labeling_log.json")

del tydiqa_raw
gc.collect()


In [ ]:
# ── Stratified 80/20 split by complexity_label ────────────────────────────
tydiqa_train, tydiqa_eval = stratified_split(
    labeled_records,
    key="complexity_label",
    ratio=TYDIQA_TRAIN_RATIO,
    seed=SEED,
)

# Overwrite split field
for rec in tydiqa_train:
    rec["split"] = "train"
for rec in tydiqa_eval:
    rec["split"] = "eval"

write_jsonl(tydiqa_train, DATA_DIR / "tydiqa_id_train.jsonl")
write_jsonl(tydiqa_eval,  DATA_DIR / "tydiqa_id_eval.jsonl")

# Verify stratification held
train_labels = collections.Counter(r["complexity_label"] for r in tydiqa_train)
eval_labels  = collections.Counter(r["complexity_label"] for r in tydiqa_eval)
print(f"  Train label dist: {dict(train_labels)}")
print(f"  Eval  label dist: {dict(eval_labels)}")
print(f"  ✓ Target train: {int(len(labeled_records)*TYDIQA_TRAIN_RATIO)}, actual: {len(tydiqa_train)}")

del tydiqa_train, tydiqa_eval, labeled_records
gc.collect()


## 6. XQuAD-ID (Parallel EN/ID)


Raw SQuAD-ID source files: [train](https://drive.google.com/uc?id=1LP0iB0Xe6nkbnSxMeclxexUfqCE9e5qH&export=download) and [validation](https://drive.google.com/uc?id=1KZE92j3Cnf7N6o0qrVplBqXV2XlGxnvo&export=download); mirrored by [SEACrowd/squad_id](https://huggingface.co/datasets/SEACrowd/squad_id).
Official XQuAD does not ship an Indonesian config, but XQuAD is a 1,190-row subset of SQuAD. We load `xquad.en` as the fixed English anchor set and join Indonesian translations from the raw SQuAD-ID source files using `id == xquad.en.id`.

**Critical invariant:** `question_id` is the original XQuAD/SQuAD id. NB07 uses it to match EN and ID queries for the same question when computing ?gap. Do not split `question_en` and `question_id_text` into separate files.

Schema:
```
{
  "question_id":      str,
  "question_en":      str,
  "question_id_text": str,
  "answer":           str,
  "context":          str,
  "dataset":          "xquad_id",
  "split":            "eval"
}
```


In [ ]:
print("Loading XQuAD EN anchors and raw Indonesian SQuAD-ID translations...")

# Official XQuAD has no Indonesian config. To build XQuAD-ID, use the 1,190
# XQuAD English rows as anchors, then join raw SQuAD-ID translations by id.
# These raw files are the source URLs used by SEACrowd/squad_id and keep the
# original SQuAD/XQuAD ids directly, unlike some derived HF mirrors.
# SEACrowd mirror: https://huggingface.co/datasets/SEACrowd/squad_id
# Raw SQuAD-ID train: https://drive.google.com/uc?id=1LP0iB0Xe6nkbnSxMeclxexUfqCE9e5qH&export=download
# Raw SQuAD-ID validation: https://drive.google.com/uc?id=1KZE92j3Cnf7N6o0qrVplBqXV2XlGxnvo&export=download
import html
import requests
from urllib.parse import parse_qs, urlparse

SQUAD_ID_GDRIVE = {
    "train": "1LP0iB0Xe6nkbnSxMeclxexUfqCE9e5qH",
    "validation": "1KZE92j3Cnf7N6o0qrVplBqXV2XlGxnvo",
}

def download_gdrive_json(file_id):
    url = "https://drive.usercontent.google.com/download"
    with requests.Session() as session:
        response = session.get(url, params={"id": file_id, "export": "download", "confirm": "t"})
        response.raise_for_status()

        token = None
        for key, value in response.cookies.items():
            if key.startswith("download_warning"):
                token = value
                break

        if token is None and "confirm=" in response.text:
            match = re.search(r"confirm=([0-9A-Za-z_\-]+)", response.text)
            if match:
                token = html.unescape(match.group(1))

        if token is None and "download_url" in response.text:
            match = re.search(r'"download_url":"([^"]+)"', response.text)
            if match:
                download_url = html.unescape(match.group(1).replace("\\u003d", "=").replace("\\u0026", "&"))
                token = parse_qs(urlparse(download_url).query).get("confirm", [None])[0]

        if token:
            response = session.get(url, params={"id": file_id, "export": "download", "confirm": token})
            response.raise_for_status()

        text = response.text.lstrip("\ufeff\n\r\t ")
        if not text.startswith(("{", "[")):
            raise RuntimeError(f"Google Drive returned non-JSON content for {file_id}: {text[:200]!r}")
        return json.loads(text)


def flatten_squad_id_json(data):
    rows = []
    for doc_id, paragraphs in data.get("paragraphs", {}).items():
        for paragraph in paragraphs:
            context = paragraph.get("context", "")
            for qa in paragraph.get("qas", []):
                answers = qa.get("indonesian_answers") or qa.get("indonesian_plausible_answers") or []
                rows.append({
                    "id": qa.get("id"),
                    "question": qa.get("question", ""),
                    "context": context,
                    "answers": answers,
                    "document_id": doc_id,
                })
    return rows


xquad_en = load_dataset("xquad", "xquad.en", split="validation", token=HF_TOKEN)
squad_id = []
for split_name, file_id in SQUAD_ID_GDRIVE.items():
    raw_split = download_gdrive_json(file_id)
    split_rows = flatten_squad_id_json(raw_split)
    squad_id.extend(split_rows)
    print(f"  SQuAD-ID {split_name}: {len(split_rows):,} rows")

print(f"  XQuAD EN anchors: {len(xquad_en):,} instances")
print(f"  SQuAD-ID raw pool: {len(squad_id):,} instances")
assert len(xquad_en) == 1_190, "XQuAD EN should contain exactly 1,190 validation instances"


In [ ]:
# Build XQuAD-ID parallel structure: one row per XQuAD question id.

def parse_answer_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            return [{"text": value}]
    if isinstance(value, dict):
        texts = value.get("text", [])
        return [{"text": t} for t in texts] if isinstance(texts, list) else [{"text": texts}]
    if isinstance(value, list):
        return value
    return []


id_map = {ex["id"]: ex for ex in squad_id if ex.get("id")}

parallel_records = []
missing_ids = []
for ex_en in xquad_en:
    question_id = ex_en["id"]
    ex_id = id_map.get(question_id)
    if ex_id is None:
        missing_ids.append(question_id)
        continue

    answers = parse_answer_list(ex_id.get("answers"))
    if not answers:
        answers = parse_answer_list(ex_en.get("answers"))

    answer_text = ""
    if answers:
        first = answers[0]
        answer_text = first.get("text", "") if isinstance(first, dict) else str(first)

    parallel_records.append({
        "question_id":      question_id,
        "qid":              question_id,  # backward-compatible alias for older NB07 code
        "question_en":      ex_en["question"],
        "question_id_text": ex_id["question"],
        "answer":           answer_text,
        "context":          ex_id["context"],
        "dataset":          "xquad_id",
        "split":            "eval",
    })

if missing_ids:
    print(f"  Dropped {len(missing_ids):,} XQuAD rows without Indonesian translation")
    print(f"  Missing examples: {missing_ids[:5]}")

assert parallel_records, "No XQuAD-ID pairs found after dropping missing translations"

write_jsonl(parallel_records, DATA_DIR / "xquad_id_parallel.jsonl")
print(f"  XQuAD-ID parallel pairs: {len(parallel_records):,}")

sample = parallel_records[0]
assert sample["question_id"], "question_id is empty"
assert sample["question_en"], "question_en is empty"
assert sample["question_id_text"], "question_id_text is empty"
assert sample["question_en"] != sample["question_id_text"], (
    "EN and ID questions are identical - join may have gone wrong"
)
print(f"  Sample question_id={sample['question_id']}")
print(f"    EN: {sample['question_en']}")
print(f"    ID: {sample['question_id_text']}")

del xquad_en, squad_id, id_map, parallel_records


## 7. Wikipedia ID — Retrieval Corpus

Used exclusively as the **open-domain retrieval corpus** for:
- NB07: Δgap measurement (Recall@10, MRR)
- NB08: RAGAS evaluation

**Why streaming?**  
The full Indonesian Wikipedia dump (`20231101.id`) has ~600K+ articles. Downloading the entire dump takes ~20 min and ~10GB of disk. Using `streaming=True` lets us pull exactly 20K articles without hitting Kaggle's disk or time limits.

**Schema per article:**
```
{
  "doc_id": str,    # "wiki_id__{url_hash}"
  "title":  str,
  "text":   str,    # full article text (truncated to first 2K chars for memory)
  "dataset": "wikipedia_id"
}
```

> Note: We truncate article text to **2,000 characters** per document. This is intentional — BGE-M3 encodes up to 8,192 tokens but extremely long documents increase indexing time significantly. 2K characters (~500 tokens) captures the lead paragraph which contains the most factual content for retrieval.


In [ ]:
import hashlib

MAX_TEXT_CHARS = 2_000  # truncate each article to first 2K chars

print(f"Streaming Wikipedia ID, collecting {WIKI_ID_TARGET:,} articles…")
print("  (This will take ~10–20 min depending on Kaggle's HF cache state)")

wiki_id_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.id",
    split="train",
    streaming=True,
)

wiki_corpus = []
seen_titles = set()

for article in wiki_id_stream:
    if len(wiki_corpus) >= WIKI_ID_TARGET:
        break

    title = article.get("title", "").strip()
    text  = article.get("text", "").strip()

    # Skip stub articles and redirects
    if not text or len(text) < 100:
        continue
    if title in seen_titles:
        continue

    seen_titles.add(title)
    # Stable doc_id from title hash
    doc_id = "wiki_id__" + hashlib.md5(title.encode("utf-8")).hexdigest()[:12]

    wiki_corpus.append({
        "doc_id":  doc_id,
        "title":   title,
        "text":    text[:MAX_TEXT_CHARS],
        "dataset": "wikipedia_id",
    })

    if len(wiki_corpus) % 2_000 == 0:
        print(f"  Collected {len(wiki_corpus):,} / {WIKI_ID_TARGET:,}…")

print(f"  ✓ Wikipedia ID corpus: {len(wiki_corpus):,} articles collected")
assert len(wiki_corpus) >= WIKI_ID_TARGET * 0.95, (
    f"Got only {len(wiki_corpus)} articles — streaming may have been truncated early"
)

write_jsonl(wiki_corpus, DATA_DIR / "wikipedia_id_corpus.jsonl")

del wiki_corpus, wiki_id_stream, seen_titles
gc.collect()


## 8. Validation — Shape & Field Checks

Verify all outputs before uploading to Kaggle Dataset.


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/kaggle/working/data")

EXPECTED = {
    "hotpotqa_train.jsonl": {
        "min_count": 9_900,
        "required_fields": {"id", "question", "answer", "lang", "dataset", "split", "complexity_label"},
        "check_values": {"lang": "en", "dataset": "hotpotqa", "complexity_label": "C"},
    },
    "hotpotqa_eval.jsonl": {
        "min_count": 7_405,
        "max_count": 7_405,
        "required_fields": {"id", "question", "answer", "lang", "dataset", "split"},
        "check_values": {"lang": "en", "dataset": "hotpotqa", "split": "eval"},
    },
    "hotpotqa_source_docs.jsonl": {
        "min_count": 1_000,  # conservative lower bound — actual ~60K+ unique docs
        "required_fields": {"doc_id", "title", "text", "dataset"},
        "check_values": {"dataset": "hotpotqa"},
    },
    "tydiqa_id_train.jsonl": {
        "min_count": 1,
        "required_fields": {"id", "question", "answer", "lang", "dataset", "split", "complexity_label"},
        "check_values": {"lang": "id", "dataset": "tydiqa_id", "split": "train"},
    },
    "tydiqa_id_eval.jsonl": {
        "min_count": 1,
        "required_fields": {"id", "question", "answer", "lang", "dataset", "split", "complexity_label"},
        "check_values": {"lang": "id", "dataset": "tydiqa_id", "split": "eval"},
    },
    "xquad_id_parallel.jsonl": {
        "min_count": 1,
        "required_fields": {"question_id", "question_en", "question_id_text", "answer", "context", "dataset", "split"},
        "check_values": {"dataset": "xquad_id", "split": "eval"},
        "extra_checks": ["parallel_integrity"],
    },
    "wikipedia_id_corpus.jsonl": {
        "min_count": int(WIKI_ID_TARGET * 0.95),
        "required_fields": {"doc_id", "title", "text", "dataset"},
        "check_values": {"dataset": "wikipedia_id"},
    },
}

all_pass = True
results_summary = {}

for filename, spec in EXPECTED.items():
    fpath = DATA_DIR / filename
    errors = []

    if not fpath.exists():
        print(f"  ✗ MISSING: {filename}")
        all_pass = False
        continue

    records = read_jsonl(fpath)
    count = len(records)

    # Count checks
    if count < spec.get("min_count", 0):
        errors.append(f"count {count} < min {spec['min_count']}")
    if "max_count" in spec and count > spec["max_count"]:
        errors.append(f"count {count} > max {spec['max_count']}")

    # Field checks (sample first + last record)
    for sample_rec in [records[0], records[-1]]:
        missing = spec["required_fields"] - set(sample_rec.keys())
        if missing:
            errors.append(f"missing fields: {missing}")

    # Value checks (spot-check first record)
    for field, expected_val in spec.get("check_values", {}).items():
        if records[0].get(field) != expected_val:
            errors.append(f"records[0][{field!r}] = {records[0].get(field)!r}, expected {expected_val!r}")

    # Parallel QA-specific integrity
    if "parallel_integrity" in spec.get("extra_checks", []):
        bad = [r for r in records if r.get("question_en") == r.get("question_id_text")]
        if bad:
            errors.append(f"{len(bad)} records have identical EN/ID questions")
        no_qid = [r for r in records if not r.get("question_id")]
        if no_qid:
            errors.append(f"{len(no_qid)} records missing question_id")

    status = "✓" if not errors else "✗"
    if errors:
        all_pass = False
    results_summary[filename] = {"count": count, "errors": errors}
    print(f"  {status} {filename}: {count:,} records" + (f"  ← {'; '.join(errors)}" if errors else ""))

print()
if all_pass:
    print("✅ All validation checks passed.")
else:
    print("❌ Some checks failed — see above.")


## 9. Summary & Next Steps

### Files produced
| File | Purpose | Used by |
|------|---------|---------|
| `hotpotqa_train.jsonl` | Classifier training (all label C) | NB02 |
| `hotpotqa_eval.jsonl` | Ablation evaluation | NB06 |
| `hotpotqa_source_docs.jsonl` | Retrieval corpus | NB03, NB06 |
| `tydiqa_id_train.jsonl` | Classifier training (labels A/B) | NB02 |
| `tydiqa_id_eval.jsonl` | Downstream evaluation | NB07, NB08 |
| `xquad_id_parallel.jsonl` | Δgap + ablation + RAGAS | NB06, NB07, NB08 |
| `wikipedia_id_corpus.jsonl` | Open-domain retrieval corpus | NB03, NB07, NB08 |
| `tydiqa_labeling_log.json` | Discard count for paper | — |

### ⚠️ Required action before starting NB02
1. In Kaggle, go to **"Save Version"** → **"Save & Run All"** to commit this notebook.
2. Upload all files from `/kaggle/working/data/` to a new Kaggle Dataset named **`crosslingual-rag-data`**.
3. In NB02–NB08, add `crosslingual-rag-data` as an input dataset — files will appear at `/kaggle/input/crosslingual-rag-data/`.


In [ ]:
# Print file sizes for reference
print("Output file sizes:")
for f in sorted(DATA_DIR.iterdir()):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name}: {size_mb:.1f} MB")

print()
print("NB01 complete. Upload /kaggle/working/data/ → Kaggle Dataset 'crosslingual-rag-data'")
